# So sánh 5 mô hình trên BRFSS 2011, 2013 và 2015

Trong Kaggle Notebook, chọn **Add Input** rồi thêm [CDC Behavioral Risk Factor Surveillance System](https://www.kaggle.com/datasets/cdc/behavioral-risk-factor-surveillance-system). Chạy **Run All** để đọc `2011.csv`, `2013.csv` và `2015.csv`.

Thí nghiệm đọc cả ba năm từ dữ liệu gốc để áp dụng cùng quy tắc làm sạch. Notebook xuất mô hình và báo cáo sang thư mục riêng. Dùng `MAX_ROWS_PER_YEAR` để chạy thử với mẫu nhỏ và đặt lại thành `None` khi huấn luyện chính thức.

Nguồn đối chiếu: [CDC 2011](https://www.cdc.gov/brfss/annual_data/2011/pdf/CODEBOOK11_LLCP.pdf), [CDC 2013](https://www.cdc.gov/brfss/annual_data/2013/pdf/codebook13_llcp.pdf), [CDC 2015](https://www.cdc.gov/brfss/annual_data/2015/pdf/codebook15_llcp.pdf).

In [ ]:
import os
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

YEARS = (2011, 2013, 2015)
FEATURES = ["Age", "BMI", "HighBP", "HighChol", "GenHlth"]
TARGET = "Diabetes_binary"
SEED = 42
THRESHOLD = 0.5
N_JOBS = min(4, os.cpu_count() or 1)

DATA_DIR = None  # Hoặc Path("đường dẫn thư mục chứa cả ba CSV")
MAX_ROWS_PER_YEAR = None  # Ví dụ 5000 để chạy thử
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = (
    Path("/kaggle/working/brfss_multiyear")
    if KAGGLE_INPUT.is_dir()
    else Path("artifacts/brfss_multiyear")
)

## 1. Chuẩn hóa dữ liệu

- Tuổi giữ mã nhóm 1–13. BMI dùng đơn vị kg/m². Sức khỏe chung giữ thang 1–5.
- Huyết áp và cholesterol chỉ giữ câu trả lời có hoặc không rõ ràng.
- Nhãn thí nghiệm là **đã được báo có tiểu đường ngoài thai kỳ**. Mã `DIABETE3=1` là dương tính. Mã `3` (không) và `4` (tiền tiểu đường) là âm tính theo định nghĩa này. Loại mã `2` (chỉ khi mang thai), mã không rõ, từ chối và giá trị thiếu.
- Năm khảo sát dùng để chia tập và báo cáo. Hàng trùng 5 feature chưa đủ để kết luận hai người là một, nên giữ các phản hồi hợp lệ.

In [ ]:
RAW_COLUMNS = {
    "_AGEG5YR": "Age",
    "_BMI5": "BMI",
    "BPHIGH4": "HighBP",
    "TOLDHI2": "HighChol",
    "GENHLTH": "GenHlth",
    "DIABETE3": TARGET,
}


def locate_data_dir():
    if DATA_DIR is None:
        root = KAGGLE_INPUT if KAGGLE_INPUT.is_dir() else Path("data")
        candidates = sorted({
            path.parent for path in root.rglob("2011.csv")
            if all((path.parent / f"{year}.csv").is_file() for year in YEARS)
        })
        if len(candidates) != 1:
            raise FileNotFoundError(
                f"Tìm thấy {len(candidates)} thư mục chứa đủ ba CSV trong {root}. "
                "Hãy thêm dataset BRFSS vào Kaggle hoặc đặt DATA_DIR."
            )
        return candidates[0]
    directory = Path(DATA_DIR)
    missing = [year for year in YEARS if not (directory / f"{year}.csv").is_file()]
    if missing:
        raise FileNotFoundError(f"Thiếu CSV cho các năm {missing} trong {directory}.")
    return directory


def load_survey(path, year):
    raw = pd.read_csv(
        path, usecols=lambda name: name.strip().upper() in RAW_COLUMNS,
        dtype="string", encoding="utf-8-sig",
    )
    raw.columns = raw.columns.str.strip().str.upper()
    missing = set(RAW_COLUMNS) - set(raw.columns)
    if missing:
        raise ValueError(f"{path.name} thiếu cột gốc: {sorted(missing)}")
    total = len(raw)
    clean = raw.apply(pd.to_numeric, errors="coerce").rename(columns=RAW_COLUMNS)
    clean["Age"] = clean["Age"].where(clean["Age"].isin(range(1, 14)))
    clean["BMI"] = clean["BMI"].where(clean["BMI"].between(1200, 9999)) / 100
    clean["HighBP"] = clean["HighBP"].map({1: 1, 3: 0})
    clean["HighChol"] = clean["HighChol"].map({1: 1, 2: 0})
    clean["GenHlth"] = clean["GenHlth"].where(clean["GenHlth"].isin(range(1, 6)))
    clean[TARGET] = clean[TARGET].map({1: 1, 3: 0, 4: 0})
    clean = clean[FEATURES + [TARGET]].dropna().astype({
        "Age": "int8", "BMI": "float32", "HighBP": "int8",
        "HighChol": "int8", "GenHlth": "int8", TARGET: "int8",
    })
    valid = len(clean)
    if MAX_ROWS_PER_YEAR is not None and valid > MAX_ROWS_PER_YEAR:
        clean, _ = train_test_split(
            clean, train_size=MAX_ROWS_PER_YEAR, stratify=clean[TARGET], random_state=SEED,
        )
    if clean[TARGET].nunique() != 2:
        raise ValueError(f"Năm {year} không có đủ hai lớp sau khi lọc.")
    clean = clean.assign(year=year)
    return clean, {
        "year": year, "raw_rows": total, "valid_rows": valid,
        "rows_used": len(clean), "positive_share": float(clean[TARGET].mean()),
    }


directory = locate_data_dir()
surveys = [load_survey(directory / f"{year}.csv", year) for year in YEARS]
data = pd.concat([frame for frame, _ in surveys], ignore_index=True)
counts = pd.DataFrame([summary for _, summary in surveys])
display(counts)

## 2. Chia dữ liệu và huấn luyện

Chia **70% train, 15% validation, 15% test**, phân tầng theo cặp năm và nhãn. Chọn mô hình theo Average Precision trên validation. Với từng mô hình, tìm ngưỡng tối đa F1 trên validation rồi giữ nguyên ngưỡng khi đánh giá test. Báo cáo cả ngưỡng 0,5 và ngưỡng đã chọn. F1 cân bằng precision và recall cho thí nghiệm này.

In [ ]:
def strata(frame):
    return frame["year"] * 10 + frame[TARGET]


train, held_out = train_test_split(
    data, test_size=0.30, stratify=strata(data), random_state=SEED,
)
validation, test = train_test_split(
    held_out, test_size=0.50, stratify=strata(held_out), random_state=SEED,
)
display(pd.DataFrame({
    "train": train.groupby("year").size(),
    "validation": validation.groupby("year").size(),
    "test": test.groupby("year").size(),
}))

models = {
    "logistic_regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=300, random_state=SEED),
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100, max_depth=12, min_samples_leaf=20, n_jobs=N_JOBS, random_state=SEED,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, max_leaf_nodes=31, early_stopping=True, random_state=SEED,
    ),
    "mlp": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(32, 16), batch_size=512, max_iter=70,
            early_stopping=True, n_iter_no_change=10, random_state=SEED,
        ),
    ),
    "gaussian_naive_bayes": GaussianNB(),
}


def select_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    total = precision[:-1] + recall[:-1]
    f1 = np.divide(
        2 * precision[:-1] * recall[:-1], total,
        out=np.zeros_like(total), where=total > 0,
    )
    best = np.flatnonzero(f1 == f1.max())[-1]
    return float(thresholds[best])


def evaluate(labels, scores, threshold=THRESHOLD):
    predicted = scores >= threshold
    return {
        "roc_auc": float(roc_auc_score(labels, scores)),
        "average_precision": float(average_precision_score(labels, scores)),
        "positive_share": float(np.mean(labels)),
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
    }


validation_rows = []
tuned_thresholds = {}
for name, model in models.items():
    start = perf_counter()
    with threadpool_limits(limits=N_JOBS):
        model.fit(train[FEATURES], train[TARGET])
        scores = model.predict_proba(validation[FEATURES])[:, 1]
    tuned_thresholds[name] = select_threshold(validation[TARGET], scores)
    elapsed = round(perf_counter() - start, 2)
    for rule, threshold in (("fixed_0.5", THRESHOLD), ("validation_f1", tuned_thresholds[name])):
        validation_rows.append({
            "model": name, "threshold_rule": rule, "threshold": threshold,
            **evaluate(validation[TARGET], scores, threshold),
            "fit_and_validation_seconds": elapsed,
        })
    print(f"Đã xong {name}", flush=True)

validation_report = pd.DataFrame(validation_rows).sort_values(
    ["average_precision", "roc_auc", "model"], ascending=[False, False, True],
)
winner = validation_report.iloc[0]["model"]
display(validation_report)

In [ ]:
## Test các mô hình sau khi đã chốt mô hình theo validation.
test_rows = []
for name, model in models.items():
    with threadpool_limits(limits=N_JOBS):
        scores = model.predict_proba(test[FEATURES])[:, 1]
    for year in ("all", *YEARS):
        mask = np.ones(len(test), dtype=bool) if year == "all" else test["year"].to_numpy() == year
        for rule, threshold in (("fixed_0.5", THRESHOLD), ("validation_f1", tuned_thresholds[name])):
            test_rows.append({
                "model": name, "year": str(year), "rows": int(mask.sum()),
                "threshold_rule": rule, "threshold": threshold,
                **evaluate(test.loc[mask, TARGET], scores[mask], threshold),
            })

test_report = pd.DataFrame(test_rows)
display(test_report[test_report["year"] == "all"].sort_values(["average_precision", "threshold_rule"], ascending=[False, True]))
display(test_report[(test_report["model"] == winner) & (test_report["year"] != "all")])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
counts.to_csv(OUTPUT_DIR / "dataset_counts.csv", index=False)
validation_report.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
test_report.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)

# Đối chứng cùng thuật toán và cùng test 2015. Chỉ thay nguồn dữ liệu huấn luyện.
train_2015 = train[train["year"] == 2015]
val_2015 = validation[validation["year"] == 2015]
test_2015 = test[test["year"] == 2015]
baseline = clone(models[winner])
with threadpool_limits(limits=N_JOBS):
    baseline.fit(train_2015[FEATURES], train_2015[TARGET])
    comparison = []
    for cohort, model, fit_rows in (
        ("2015_only", baseline, len(train_2015)),
        ("2011_2013_2015", models[winner], len(train)),
    ):
        threshold = select_threshold(val_2015[TARGET], model.predict_proba(val_2015[FEATURES])[:, 1])
        scores = model.predict_proba(test_2015[FEATURES])[:, 1]
        comparison.append({
            "training_cohort": cohort, "model": winner, "train_rows": fit_rows,
            "test_rows": len(test_2015), "threshold": threshold,
            **evaluate(test_2015[TARGET], scores, threshold),
        })
comparison_report = pd.DataFrame(comparison)
display(comparison_report)
comparison_report.to_csv(OUTPUT_DIR / "cohort_comparison_2015.csv", index=False)

winner_test = next(row for row in test_rows if row["model"] == winner and row["year"] == "all" and row["threshold_rule"] == "validation_f1")
joblib.dump({
    "pipeline": models[winner],
    "features": FEATURES,
    "threshold": tuned_thresholds[winner],
    "model_version": f"brfss_multiyear_{winner}_v1",
    "metrics": {key: value for key, value in winner_test.items() if key not in {"model", "year", "rows", "threshold_rule", "threshold"}},
    "survey_years": YEARS,
    "target_definition": "DIABETE3 1 positive, 3 and 4 negative, other codes excluded",
    "sklearn_version": sklearn.__version__,
    "max_rows_per_year": MAX_ROWS_PER_YEAR,
}, OUTPUT_DIR / "brfss_multiyear_best.bin")
print(f"Mô hình được chọn theo validation: {winner}")
print(f"scikit-learn {sklearn.__version__} | Kết quả: {OUTPUT_DIR.resolve()}")

**Đọc kết quả:** So sánh recall, precision và F1 trên test giữa hai ngưỡng. Đổi ngưỡng không đổi ROC-AUC hoặc Average Precision. So sánh Average Precision với `positive_share`, mốc tham chiếu của cách xếp hạng ngẫu nhiên. Dùng `cohort_comparison_2015.csv` để xem việc gộp thêm năm có giúp trên cùng test 2015 trong thí nghiệm này hay không.

Mô hình xuất từ notebook lưu phiên bản scikit-learn và ngưỡng đánh giá. API hiện tại vẫn gán nhãn bằng ngưỡng 0,5. Nếu sau này muốn dùng trong API, hãy huấn luyện và xuất bằng scikit-learn 1.6.1 để tương thích.